# Credit Scoring - Loan Approval Prediction

CodeAlpha ML Internship - Task 1

Goal: predict if a loan application gets approved (`Y`) or rejected (`N`) using the applicant's financial details.

## 1. Imports

In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report, confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## 2. Load the dataset

In [2]:
data = pd.read_csv("Loan Eligibility Prediction.csv")
data.head()

   Customer_ID  Gender Married  Dependents     Education Self_Employed  Applicant_Income  Coapplicant_Income  Loan_Amount  Loan_Amount_Term  Credit_History Property_Area Loan_Status
0          569  Female      No           0      Graduate            No              2378                 0.0            9               360               1         Urban           N
1           15    Male     Yes           2      Graduate            No              1299              1086.0           17               120               1         Urban           Y
2           95    Male      No           0  Not Graduate            No              3620                 0.0           25               120               1     Semiurban           Y
3          134    Male     Yes           0      Graduate           Yes              3459                 0.0           25               120               1     Semiurban           Y
4          556    Male     Yes           1      Graduate            No              5468  

## 3. First look at the data

In [3]:
print("Rows and columns:", data.shape)
print("\nColumn types:")
print(data.dtypes)
print("\nMissing values:")
print(data.isna().sum())
print("\nDuplicate rows:", data.duplicated().sum())

Rows and columns: (614, 13)

Column types:
Customer_ID             int64
Gender                    str
Married                   str
Dependents              int64
Education                 str
Self_Employed             str
Applicant_Income        int64
Coapplicant_Income    float64
Loan_Amount             int64
Loan_Amount_Term        int64
Credit_History          int64
Property_Area             str
Loan_Status               str
dtype: object

Missing values:
Customer_ID           0
Gender                0
Married               0
Dependents            0
Education             0
Self_Employed         0
Applicant_Income      0
Coapplicant_Income    0
Loan_Amount           0
Loan_Amount_Term      0
Credit_History        0
Property_Area         0
Loan_Status           0
dtype: int64

Duplicate rows: 0


## 4. Quick EDA

Before training I wanted to see which columns actually matter. Credit history looks like the biggest one.

In [4]:
print("Approved vs rejected:")
print(data["Loan_Status"].value_counts())

# approval rate for each group
for col in ["Credit_History", "Education", "Married", "Property_Area"]:
    print(f"\nApproval rate by {col}:")
    print((data.groupby(col)["Loan_Status"].apply(lambda s: (s == "Y").mean()) * 100).round(1))

Approved vs rejected:
Loan_Status
Y    422
N    192
Name: count, dtype: int64

Approval rate by Credit_History:
Credit_History
0    10.9
1    78.9
Name: Loan_Status, dtype: float64

Approval rate by Education:
Education
Graduate        70.8
Not Graduate    61.2
Name: Loan_Status, dtype: float64

Approval rate by Married:
Married
No     63.3
Yes    71.7
Name: Loan_Status, dtype: float64

Approval rate by Property_Area:
Property_Area
Rural        61.5
Semiurban    76.8
Urban        65.8
Name: Loan_Status, dtype: float64


## 5. Features and target

`Loan_Status` is what we predict. `Customer_ID` is just an ID number, it says nothing about the applicant, so I drop it.

In [5]:
data = data.drop_duplicates().copy()
data = data.drop(columns=["Customer_ID"])
data = data.dropna(subset=["Loan_Status"])

X = data.drop(columns="Loan_Status")
y = data["Loan_Status"]

print("Input columns:", list(X.columns))

Input columns: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Applicant_Income', 'Coapplicant_Income', 'Loan_Amount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']


## 6. Train / test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 491
Test rows: 123


## 7. Preprocessing

Numbers: fill missing with median, then scale.
Text columns: fill missing with most common value, then one-hot encode.

In [7]:
numeric_columns = X.select_dtypes(include="number").columns
categorical_columns = X.select_dtypes(exclude="number").columns

numeric_steps = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_steps = Pipeline(steps=[
    ("fill_missing", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

prepare_columns = ColumnTransformer(transformers=[
    ("numbers", numeric_steps, numeric_columns),
    ("categories", categorical_steps, categorical_columns),
])

## 8. Train 3 models

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

trained_models = {}
for model_name, model in models.items():
    model_pipeline = Pipeline(steps=[
        ("prepare", prepare_columns),
        ("classifier", model),
    ])
    model_pipeline.fit(X_train, y_train)
    trained_models[model_name] = model_pipeline
    print(model_name, "trained")

Logistic Regression trained
Decision Tree trained
Random Forest trained


## 9. Compare results

`Y` (approved) is treated as the positive class. ROC-AUC uses the predicted probability of `Y`.

In [9]:
results = []
for model_name, model_pipeline in trained_models.items():
    predictions = model_pipeline.predict(X_test)
    y_index = list(model_pipeline.classes_).index("Y")
    prob_yes = model_pipeline.predict_proba(X_test)[:, y_index]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, pos_label="Y"),
        "Recall": recall_score(y_test, predictions, pos_label="Y"),
        "F1": f1_score(y_test, predictions, pos_label="Y"),
        "ROC-AUC": roc_auc_score(y_test == "Y", prob_yes),
    })

results_table = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).round(4)
results_table

                 Model  Accuracy  Precision  Recall      F1  ROC-AUC
0  Logistic Regression    0.8049     0.8020  0.9529  0.8710   0.8012
2        Random Forest    0.7805     0.7959  0.9176  0.8525   0.7861
1        Decision Tree    0.8049     0.8144  0.9294  0.8681   0.6735


## 10. Best model in detail

In [10]:
best_name = results_table.iloc[0]["Model"]
best_model = trained_models[best_name]
best_predictions = best_model.predict(X_test)

print("Best model (by ROC-AUC):", best_name)
print("\nConfusion matrix (rows = actual N, Y):")
print(confusion_matrix(y_test, best_predictions, labels=["N", "Y"]))
print("\nClassification report:")
print(classification_report(y_test, best_predictions, zero_division=0))

Best model (by ROC-AUC): Logistic Regression

Confusion matrix (rows = actual N, Y):
[[18 20]
 [ 4 81]]

Classification report:
              precision    recall  f1-score   support

           N       0.82      0.47      0.60        38
           Y       0.80      0.95      0.87        85

    accuracy                           0.80       123
   macro avg       0.81      0.71      0.74       123
weighted avg       0.81      0.80      0.79       123



## 11. Try one prediction

In [11]:
example_application = X_test.iloc[[0]]
print("Applicant details:")
display(example_application)
print("Predicted loan status:", best_model.predict(example_application)[0])
print("Actual loan status:", y_test.iloc[0])

Applicant details:
     Gender Married  Dependents Education Self_Employed  Applicant_Income  Coapplicant_Income  Loan_Amount  Loan_Amount_Term  Credit_History Property_Area
158  Female      No           0  Graduate            No              1500              1800.0          103               360               0     Semiurban
Predicted loan status: N
Actual loan status: N
